# 07 · 최종 제출 파일 점검 (CPU)

고른 제출 파일(CSV)마다 다음을 확인합니다. GPU가 필요 없습니다.
1. **형식**: `id, answer` 두 열, `sample_submission.csv`와 같은 6,714문항·같은 순서, 답은 a~d 한 글자, 빈칸 없음.
2. **재현**: 같은 이름의 설정 JSON(조합 노트북이 저장)이 있으면, JSON에 적힌 멤버 예측 파일과 규칙만으로 **이 노트북의 독립 코드가 test 답을 다시 만들어** 저장된 CSV와 같은지 봅니다.
   - `05_ens_search` / 수동 조합 JSON: 멤버별 test 확률 → (온도 보정) → 가중 평균 → 최댓값. 멤버 파일의 md5가 JSON 기록과 같은지도 봅니다.
   - `06_anchor_rescue` JSON: 기준 모델 확률(온도 보정) + 보조 모델 표 → 네 조건(불일치·확신도·마진·다수 동의)을 모두 만족할 때만 뒤집기.
3. **비교**: 고른 파일끼리 test 답이 다른 문항 수 (공개 리더보드는 test의 약 절반이라 다른 문항의 절반쯤이 공개 점수에 반영됨).
4. (선택) **다수결 파일**: 입력 CSV 여러 개의 다수결을 다시 만들어 저장된 파일과 비교 (동률은 알파벳 앞 글자).

## 1. 설정 — 제출할 파일 경로 (OUT_DIR 기준)

In [ ]:
DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
CANDIDATES = {                                    # 이름: OUT_DIR 기준 CSV 경로 (같은 이름의 .json이 있으면 재현 점검)
    "제출 1": "ens_search/submission_my_search_ea0e9f.csv",
    "제출 2": None,                               # 예: "anchor_rescue/submission_anchor_…csv"
}
MAJORITY = None    # 선택: {"saved": "ensembles/submission_majority_vote.csv", "inputs": ["ensembles/a.csv", "ensembles/b.csv", "ensembles/c.csv"]}
N_PUBLIC = 3357    # 공개 리더보드 문항 수 (점수 × 3,357 = 맞힌 문항 수)

## 2. 데이터 (CSV만)

In [ ]:
import os, re, json, math, hashlib, subprocess
import numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
if not os.path.exists(os.path.join(DATA_DIR, "sample_submission.csv")):
    r = subprocess.run(["7z", "e", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP, "sample_submission.csv", "test.csv", "-r"], capture_output=True, text=True)
    assert os.path.exists(os.path.join(DATA_DIR, "sample_submission.csv")), r.stdout[-500:] + r.stderr[-500:]
SS = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv")); TEST_IDS = SS.id.tolist(); LETTERS = "abcd"
print(f"sample_submission {len(SS):,}문항")
def md5file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
def P_of(rel):
    d = pd.read_csv(os.path.join(OUT_DIR, rel)).drop_duplicates("id", keep="last").set_index("id")
    miss = [i for i in TEST_IDS if i not in d.index]; assert not miss, f"{rel}: test 문항 {len(miss)}개 없음"
    return d.loc[TEST_IDS, ["p_a", "p_b", "p_c", "p_d"]].to_numpy(float)
def temp_scale(p, T):
    q = np.clip(p, 1e-12, 1.0) ** (1.0 / T); return q / q.sum(1, keepdims=True)
def answers_of(path):
    s = pd.read_csv(path).set_index("id").reindex(TEST_IDS)["answer"].astype(str).str.strip().str.lower()
    return s.map({L: i for i, L in enumerate(LETTERS)}).to_numpy()

## 3. 점검 함수 (형식 · 조합 재현 · 교체 규칙 재현)

In [ ]:
def check_format(path):
    d = pd.read_csv(path, dtype=str); probs = []
    if list(d.columns) != ["id", "answer"]: probs.append(f"열 이름 {list(d.columns)} (id, answer여야 함)")
    if len(d) != len(SS): probs.append(f"행 수 {len(d)} (≠ {len(SS)})")
    elif list(d.id) != TEST_IDS: probs.append("id 순서가 sample_submission과 다름")
    if "answer" in d:
        bad = d.answer.isna() | ~d.answer.astype(str).isin(list(LETTERS))
        if bad.any(): probs.append(f"a~d가 아닌 답 {int(bad.sum())}개")
    return probs

def rebuild_ens(info):
    # 05_ens_search / 수동 조합 JSON: {"cfg": {"weights", "method", "temp", "full"}, "members": {실행: {"test", "T", "test_md5", "full_test"}}}
    c, mem = info["cfg"], info["members"]; runs = list(c["weights"]); notes = []
    Ps = []
    for r in runs:
        m = mem[r]; fp = os.path.join(OUT_DIR, m["test"])
        if "test_md5" in m and md5file(fp) != m["test_md5"]: notes.append(f"{r} test 파일 md5가 JSON 기록과 다름")
        P = P_of(m["test"])
        if c.get("full", "none") != "none" and "full_test" in m:
            Pf = P_of(m["full_test"]); P = (P + Pf) / 2 if c["full"] == "mix" else Pf
        Ps.append(temp_scale(P, m["T"]) if c.get("temp") == "val" else P)
    w = np.array([float(c["weights"][r]) for r in runs]); w = w / w.sum()
    if c["method"] == "prob": S = sum(wi * p for wi, p in zip(w, Ps))
    elif c["method"] == "logprob": S = sum(wi * np.log(np.clip(p, 1e-6, 1.0)) for wi, p in zip(w, Ps))
    else: S = sum(wi * np.eye(4)[p.argmax(1)] for wi, p in zip(w, Ps)) + 1e-3 * sum(wi * p for wi, p in zip(w, Ps))
    desc = " + ".join(f"{r}×{c['weights'][r]:g}" for r in runs) + f" | {c['method']} · 온도 {c.get('temp', 'none')} · full {c.get('full', 'none')}"
    return S.argmax(1), desc, notes

def rebuild_anchor(meta):
    # 06_anchor_rescue JSON: 기준 모델 확률(온도 보정) + 보조 모델 표 → 네 조건을 모두 만족할 때만 다른 보기로
    rule, files = meta["rule"], meta["files"]
    PA = P_of(files[meta["anchor"]]["test"])
    PA = temp_scale(PA, float(rule["T_anchor"]))                               # 보정 안 한 실행은 T = 1.0으로 기록됨 (06번과 같은 계산)
    RP = np.stack([P_of(files[r]["test"]) for r in meta["rescuers"]])          # (보조 모델 수, 문항, 4)
    n = PA.shape[0]; a = PA.argmax(1)
    votes = np.zeros((n, 4)); psum = RP.sum(0)
    for m in range(RP.shape[0]): votes[np.arange(n), RP[m].argmax(1)] += 1
    s = np.sort(PA, 1); pmax, marg = s[:, -1], s[:, -1] - s[:, -2]
    v = votes.copy(); v[np.arange(n), a] = -1                                   # 기준 답을 뺀 다른 보기 중 가장 많은 표 (동률은 확률 합)
    alt = (v + 1e-3 * psum / max(psum.max(), 1e-9)).argmax(1); cnt = v[np.arange(n), alt]
    flip = (cnt >= rule["AGREE"]) & (pmax < rule["CONF"]) & (marg < rule["MARGIN"])
    desc = (f"기준 {meta['anchor']} · 보조 {len(meta['rescuers'])}개 (판정 {len(meta.get('judges') or [])}) | CONF < {rule['CONF']} · MARGIN < {rule['MARGIN']} · "
            f"AGREE ≥ {rule['AGREE']} · T {float(rule['T_anchor']):.2f} | 뒤집음 {int(flip.sum())} (JSON 기록 {meta.get('test_flips')})")
    notes = [] if meta.get("test_flips") in (None, int(flip.sum())) else ["뒤집은 문항 수가 JSON 기록과 다름"]
    return np.where(flip, alt, a), desc, notes

## 4. 후보별 점검

In [ ]:
ANS, ROWS = {}, []
for name, rel in CANDIDATES.items():
    if not rel: continue
    p = os.path.join(OUT_DIR, rel)
    if not os.path.exists(p): print(f"❌ {name}: 파일 없음 {p}"); continue
    probs = check_format(p); a = answers_of(p); ANS[name] = a
    row = {"후보": name, "파일": rel, "형식": "✅" if not probs else "❌ " + "; ".join(probs), "md5": md5file(p)[:12],
           "답 분포": dict(zip(LETTERS, np.bincount(a[~np.isnan(a)].astype(int), minlength=4).tolist())) if not probs else "-"}
    jp = p[:-4] + ".json"
    if os.path.exists(jp):
        info = json.load(open(jp, encoding="utf-8"))
        kind = "교체 규칙" if "anchor" in info else ("조합" if "cfg" in info else None)
        if kind:
            re_a, desc, notes = (rebuild_anchor if kind == "교체 규칙" else rebuild_ens)(info)
            same = int((re_a != a).sum())
            row["재현"] = ("✅ 같음" if same == 0 else f"❌ {same}문항 다름") + (" · ⚠️ " + "; ".join(notes) if notes else "")
            print(f"▶ {name} [{kind}] {desc}")
            if kind == "조합" and info.get("val_acc") is not None: print(f"   JSON 기록: 검증 {info['val_acc']:.2f}% · 오답 {info.get('val_err')}")
            if kind == "교체 규칙": print(f"   JSON 기록: 교차 적합 살림 {info['cv']['rescue']} · 망침 {info['cv']['harm']} · 채택 {info['cv']['adopt']}")
        else: row["재현"] = "JSON 형식 모름"
    else: row["재현"] = "설정 JSON 없음 (재현 점검 생략)"
    ROWS.append(row)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
print(pd.DataFrame(ROWS).to_string(index=False))
names = list(ANS)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        d = int((ANS[names[i]] != ANS[names[j]]).sum())
        print(f"\n{names[i]} vs {names[j]}: test 답이 다른 문항 {d}개 → 공개 리더보드에서는 약 {d / 2:.0f}문항이 갈림 (운만으로 ±{2 * math.sqrt(max(d, 1) / 2):.0f}문항 정도는 흔함)")

## 5. (선택) 다수결 파일 다시 만들기

In [ ]:
if MAJORITY:
    A = np.stack([answers_of(os.path.join(OUT_DIR, f)) for f in MAJORITY["inputs"]], 1)
    cnt = np.stack([(A == k).sum(1) for k in range(4)], 1); maj = cnt.argmax(1)                # 동률(셋 다 다름)이면 알파벳 앞 글자
    sv = answers_of(os.path.join(OUT_DIR, MAJORITY["saved"]))
    print(f"다수결 {len(MAJORITY['inputs'])}개 입력 → 저장된 {MAJORITY['saved']}와 " + ("완전히 같음 ✅" if (maj == sv).all() else f"{int((maj != sv).sum())}문항 다름 ❌"))
else:
    print("MAJORITY = None → 건너뜀")